# From Table to Network: the Python side

CIS 568 / DSC 530, Week 11. This notebook transforms the IMDB Top 1000 table into a movie similarity network and exports it as `output.json` for the D3 force layout in `index.html` / `network.js`.

Table → one-hot vectors → similarity matrix → network → JSON

In [1]:
import json

import pandas as pd
# Install scikit-learn if it is not already available in this notebook environment
# !pip install scikit-learn

from sklearn.metrics.pairwise import cosine_similarity

datafile = "./imdb_top_1000.csv"

## Load the data

In [2]:
df = pd.read_csv(datafile)
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [3]:
df.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='str')

## Step 1: Encode (one-hot)

Each distinct value in a column becomes its own 0/1 column. Every movie ends up with exactly six 1s, one per original column.

In [4]:
# One 0/1 column per distinct value in each column
cols = ['Director', 'Genre', 'Star1', 'Star2', 'Star3', 'Star4']
df_hotbit = pd.get_dummies(df[cols], prefix='Category')

len(df_hotbit.columns)   # 4081

4081

## Step 2: Compare (cosine similarity)

With six 1s per movie, cosine similarity is simply *shared attributes / 6*.

In [5]:
similarity_matrix = cosine_similarity(df_hotbit)   # 1000 x 1000
similarity_matrix.shape

(1000, 1000)

## Step 3: Filter into a network

A threshold of 0.3 keeps pairs of movies that share at least two attributes.

In [6]:
threshold = 0.3
links = []
for i in range(len(similarity_matrix)):
    for j in range(i + 1, len(similarity_matrix)):   # upper triangle only
        if similarity_matrix[i][j] >= threshold:
            links.append({"source": i, "target": j,
                          "weight": round(similarity_matrix[i][j], 3)})

In [7]:
# Keep only movies that appear in at least one link
nodes_set = {l["source"] for l in links} | {l["target"] for l in links}

nodes = [{"id": idx,
          "title":    df['Series_Title'].iloc[idx],
          "Director": df['Director'].iloc[idx],
          "Star1":    df['Star1'].iloc[idx],
          "Star2":    df['Star2'].iloc[idx],
          "Star3":    df['Star3'].iloc[idx],
          "Star4":    df['Star4'].iloc[idx],
          "Genre":    df['Genre'].iloc[idx],
          "poster":   df['Poster_Link'].iloc[idx]}
         for idx in sorted(nodes_set)]

len(nodes), len(links)   # (326, 323)

(326, 323)

## Step 4: Export

`output.json` is the contract with the D3 side: two lists, `nodes` and `links`.

In [8]:
with open('output.json', 'w') as f:
    json.dump({"links": links, "nodes": nodes}, f, indent=4)